In [ ]:
# Stage-2 step-200 A100 inference-only visual audit. Runtime -> Run all.
from pathlib import Path
import importlib.util, json, os, re, subprocess, sys
REPOSITORY_URL = 'https://github.com/GuillermoTafoya/MRIxFields.git'
TRAINING_EVIDENCE_COMMIT = '82633d66e5ea47f96b149ea22cc192fcf4526f06'
AUDIT_IMPLEMENTATION_COMMIT = '__AUDIT_IMPLEMENTATION_COMMIT__'
if re.fullmatch(r'[0-9a-f]{40}', AUDIT_IMPLEMENTATION_COMMIT) is None:
    raise RuntimeError('Notebook is unsealed: audit implementation commit is not pinned.')
def standard_library_a100_gate():
    failure = {'stage': 'standard_library_a100_80gb_gate', 'status': 'fail', 'pip_install_invoked': False, 'dependency_download_invoked': False, 'model_weight_download_invoked': False, 'drive_mount_invoked': False, 'bank_accessed': False, 'checkpoint_loaded': False, 'private_data_accessed': False, 'inference_invoked': False, 'training_invoked': False}
    visibility = os.environ.get('CUDA_VISIBLE_DEVICES')
    if visibility is not None and visibility.strip().lower() in {'', '-1', 'none', 'nodevfiles'}:
        print(json.dumps(failure, sort_keys=True), flush=True); raise RuntimeError('CUDA visibility is disabled; attach NVIDIA A100 80 GB and rerun this sealed notebook.')
    try: probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'], text=True, capture_output=True, timeout=15)
    except (OSError, subprocess.TimeoutExpired):
        print(json.dumps(failure, sort_keys=True), flush=True); raise RuntimeError('NVIDIA A100 80 GB hardware probe failed before any external action.')
    rows = [row.strip() for row in probe.stdout.splitlines() if row.strip()]
    if probe.returncode != 0 or len(rows) != 1:
        print(json.dumps(failure, sort_keys=True), flush=True); raise RuntimeError('Exactly one visible NVIDIA A100 80 GB is required before any external action.')
    fields = [field.strip() for field in rows[0].rsplit(',', 2)]
    try: memory_mib, free_memory_mib = int(fields[1]), int(fields[2])
    except (IndexError, ValueError): memory_mib, free_memory_mib = 0, 0
    if len(fields) != 3 or 'NVIDIA A100' not in fields[0] or memory_mib < 79 * 1024 or free_memory_mib < 75 * 1024:
        print(json.dumps(failure, sort_keys=True), flush=True); raise RuntimeError('The first owner qualification requires an NVIDIA A100 80 GB runtime.')
    receipt = {'stage': 'standard_library_a100_80gb_gate', 'status': 'pass', 'gpu_name': fields[0], 'gpu_total_memory_mib': memory_mib, 'gpu_free_memory_mib': free_memory_mib, 'sealed_audit_implementation_commit': AUDIT_IMPLEMENTATION_COMMIT, 'pip_install_invoked': False, 'drive_mount_invoked': False, 'private_data_accessed': False, 'training_invoked': False}
    print(json.dumps(receipt, sort_keys=True), flush=True); return receipt
A100_HARDWARE_GATE = standard_library_a100_gate()
REPO_DIR = Path('/content/MRIxFields-stage2-step200-audit-v9-' + AUDIT_IMPLEMENTATION_COMMIT[:12])
def git_probe(repo_dir, *args):
    env = os.environ.copy(); env['GIT_OPTIONAL_LOCKS'] = '0'
    return subprocess.run(['git', *args], cwd=repo_dir, text=True, capture_output=True, env=env)
def validate_checkout(repo_dir):
    repo_dir = Path(repo_dir)
    if not repo_dir.is_dir(): raise RuntimeError('Existing audit checkout is not a directory.')
    def text(*args):
        result = git_probe(repo_dir, *args)
        if result.returncode: raise RuntimeError({'read_only_git_check_failed': args, 'returncode': result.returncode})
        return result.stdout.strip()
    if text('rev-parse', '--is-inside-work-tree') != 'true': raise RuntimeError('Existing path is not a Git worktree.')
    if Path(text('rev-parse', '--show-toplevel')).resolve() != repo_dir.resolve(): raise RuntimeError('Unexpected checkout root.')
    if text('remote').splitlines() != ['origin'] or text('remote', 'get-url', '--all', 'origin').splitlines() != [REPOSITORY_URL]: raise RuntimeError('Audit checkout origin changed.')
    if text('rev-parse', 'HEAD') != AUDIT_IMPLEMENTATION_COMMIT: raise RuntimeError('Existing audit checkout is at the wrong commit.')
    symbolic = git_probe(repo_dir, 'symbolic-ref', '-q', 'HEAD')
    if symbolic.returncode != 1 or symbolic.stdout.strip(): raise RuntimeError('Audit checkout must be detached.')
    if text('status', '--porcelain=v1', '--untracked-files=all'): raise RuntimeError('Existing audit checkout is dirty.')
    if git_probe(repo_dir, 'merge-base', '--is-ancestor', TRAINING_EVIDENCE_COMMIT, 'HEAD').returncode: raise RuntimeError('Audit commit does not descend from training evidence.')
    changed = text('diff', '--name-only', TRAINING_EVIDENCE_COMMIT, 'HEAD').splitlines()
    allowed_exact = {'src/fieldbridge/evaluation/mrixfields2026_official.py', 'src/fieldbridge/evaluation/stage2_unified_gate01_p0006.py', 'src/fieldbridge/evaluation/stage2_unified_preflight.py', 'src/fieldbridge/evaluation/stage2_step200_pilot_audit.py', 'src/fieldbridge/evaluation/stage2_step200_inference_audit.py', 'src/fieldbridge/evaluation/stage2_step200_lpips_audit.py'}
    allowed_prefixes = ('notebooks/', 'tests/', 'docs/')
    disallowed = [path for path in changed if path not in allowed_exact and not path.startswith(allowed_prefixes)]
    if disallowed: raise RuntimeError({'audit_diff_outside_restricted_scope': disallowed})
    protected = ('src/fieldbridge/models/', 'src/fieldbridge/training/', 'src/fieldbridge/data/', 'configs/')
    if any(path == 'pyproject.toml' or path.startswith(protected) for path in changed): raise RuntimeError('Protected training/model/data/config/package objects changed.')
    return {'checkout_reused_without_mutation': True, 'changed_path_count': len(changed)}
checkout_reused = REPO_DIR.exists()
if checkout_reused:
    checkout_state = validate_checkout(REPO_DIR)
else:
    subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', 'fetch', 'origin', AUDIT_IMPLEMENTATION_COMMIT], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', AUDIT_IMPLEMENTATION_COMMIT], cwd=REPO_DIR, check=True)
    checkout_state = validate_checkout(REPO_DIR); checkout_state['checkout_reused_without_mutation'] = False
environment_path = REPO_DIR / 'notebooks/stage2_step200_inference_audit_environment.py'
environment_spec = importlib.util.spec_from_file_location('stage2_step200_inference_audit_environment', environment_path)
if environment_spec is None or environment_spec.loader is None: raise RuntimeError('Sealed dependency bootstrap is unavailable.')
environment_module = importlib.util.module_from_spec(environment_spec); environment_spec.loader.exec_module(environment_module)
AUDIT_DEPENDENCY_PROVENANCE = environment_module.prepare_locked_environment(REPO_DIR / 'notebooks/stage2_step200_inference_audit_dependency_lock.json')
checkout_state = validate_checkout(REPO_DIR)
def git_text(*args):
    result = git_probe(REPO_DIR, *args)
    if result.returncode: raise RuntimeError({'read_only_git_check_failed': args, 'returncode': result.returncode})
    return result.stdout.strip()
print({'audit_implementation_commit': AUDIT_IMPLEMENTATION_COMMIT, 'training_evidence_commit': TRAINING_EVIDENCE_COMMIT, 'detached_clean_checkout': True, 'training_critical_objects_unchanged': True, 'checkout_reused_without_mutation': checkout_reused, 'runtime_requirement': 'NVIDIA A100 80 GB', 'audit_role': 'inference_only', 'dependency_lock_file_sha256': AUDIT_DEPENDENCY_PROVENANCE['lock_file_sha256'], 'dependency_download_observed': AUDIT_DEPENDENCY_PROVENANCE['dependency_download_observed']}, flush=True)
operator = REPO_DIR / 'notebooks/stage2_step200_inference_audit_operator.py'
exec(compile(operator.read_text(encoding='utf-8'), str(operator), 'exec'), globals())
